# Datamart preflight checks (Feb 2026)

In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

In [ ]:
report_month = '2026-02-01'
month_start = '2026-02-01'
month_end = '2026-02-28'

excel_path = '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx'
excel_inn_col = 'ИНН'
excel_header = 0

sa_type = 'SA'
mem_limit = '8g'
output_dir = '/home/jovyan/documents/Equaring/Проверки/outputs/datamart_preflight_feb'

In [ ]:
def normalize_numeric_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r'[eE][+-]?\d+$', s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r'\.0+$', '', s)
    s = re.sub(r'\D', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = '0' + s
    elif len(s) == 11:
        s = '0' + s
    return s


def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip().lower()
    s = re.sub(r'\s+', '', s)
    return s if s else None

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

In [ ]:
excel_raw = pd.read_excel(excel_path, header=excel_header)
excel_raw.columns = [str(c).strip() for c in excel_raw.columns]

if excel_inn_col not in excel_raw.columns:
    raise ValueError(f'Missing Excel INN column: {excel_inn_col}')

excel_df = excel_raw[[excel_inn_col]].copy()
excel_df.columns = ['inn_raw']
excel_df['inn'] = excel_df['inn_raw'].apply(normalize_numeric_str)
excel_df = excel_df[(excel_df['inn'].notna()) & (excel_df['inn'] != '')].copy()
excel_df = excel_df[['inn']].drop_duplicates().sort_values('inn').reset_index(drop=True)

excel_inn_set = set(excel_df['inn'].tolist())
len(excel_inn_set)

In [ ]:
sql = f"""
with agr as (
  select distinct
    regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.c_agr_number as string) as agreement_contract_number
  from ods_alpha.scd1_agreements a
  join ods_alpha.scd1_companies c
    on c.n_cmp = a.n_cmp_client
  where upper(trim(cast(a.acq_class as string))) = '{sa_type}'
    and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
    and coalesce(a.ods_deleted_flg, '0') <> '1'
    and coalesce(c.ods_deleted_flg, '0') <> '1'
    and c.c_inn is not null
),
r2 as (
  select
    cast(id as string) as r2_id,
    cast(c_cl_org as string) as cft_id,
    cast(c_code_in_pr as string) as c_code_in_pr
  from ods.scd1_z_r2_ip_merchants
),
dog as (
  select
    cast(id as string) as dog_id,
    cast(c_parent_id as string) as dog_parent_id,
    cast(c_code_in_pr as string) as dog_code_in_pr,
    cast(c_num as string) as dog_contract_number
  from ods.scd1_z_r2_ip_merch_dog
),
ocrm as (
  select
    regexp_replace(trim(cast(se.x_inn as string)), '[^0-9]', '') as inn,
    cast(se.row_id as string) as row_id,
    row_number() over (
      partition by regexp_replace(trim(cast(se.x_inn as string)), '[^0-9]', '')
      order by cast(se.last_upd as timestamp) desc,
               cast(se.created as timestamp) desc,
               cast(se.row_id as string) desc
    ) as rn
  from ocrm_ul.s_org_ext se
  where se.x_removed_flg = 'N'
    and se.x_duplicate_flg = 'N'
),
inn_to_cdi as (
  select
    o.inn,
    cast(e.party_id as string) as cdi_id
  from ocrm o
  left join cdiul.ext_id_org e
    on cast(e.cmo_ext_party_source_id as string) = o.row_id
   and upper(e.cmo_ext_source_system) like 'OCRM%'
  where o.rn = 1
),
cdi_to_cft as (
  select distinct
    cast(party_id as string) as cdi_id,
    cast(cmo_ext_party_source_id as string) as cft_id
  from cdiul.ext_id_org
  where upper(cmo_ext_source_system) like 'CFT%'
)
select
  agr.inn,
  agr.agr_id,
  r2.r2_id,
  agr.agreement_contract_number,
  dog.dog_contract_number,
  case
    when dog.dog_parent_id = r2.r2_id then 'join_by_parent_id'
    when dog.dog_id = r2.r2_id then 'join_by_id'
    when dog.dog_code_in_pr is not null and r2.c_code_in_pr is not null and dog.dog_code_in_pr = r2.c_code_in_pr then 'join_by_code_in_pr'
    else 'no_match'
  end as join_type,
  coalesce(dog.dog_contract_number, agr.agreement_contract_number) as contract_number_final,
  inn_to_cdi.cdi_id,
  cdi_to_cft.cft_id,
  coalesce(agr.agr_id, r2.r2_id) as agr_id_final
from agr
left join r2
  on r2.r2_id = agr.agr_id
left join dog
  on dog.dog_parent_id = r2.r2_id
  or dog.dog_id = r2.r2_id
  or (dog.dog_code_in_pr is not null and r2.c_code_in_pr is not null and dog.dog_code_in_pr = r2.c_code_in_pr)
left join inn_to_cdi
  on inn_to_cdi.inn = agr.inn
left join cdi_to_cft
  on cdi_to_cft.cdi_id = inn_to_cdi.cdi_id
"""

with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    base_df = imp.fetch(sql)

base_df['inn'] = base_df['inn'].apply(normalize_numeric_str)
base_df['agreement_contract_number'] = base_df['agreement_contract_number'].apply(normalize_contract)
base_df['dog_contract_number'] = base_df['dog_contract_number'].apply(normalize_contract)
base_df['contract_number_final'] = base_df['contract_number_final'].apply(normalize_contract)
base_df = base_df.drop_duplicates()

len(base_df)

In [ ]:
lake_inn_set = set(base_df['inn'].dropna().tolist())
in_both_set = excel_inn_set & lake_inn_set
only_excel_set = excel_inn_set - lake_inn_set
only_lake_set = lake_inn_set - excel_inn_set

fallback_df = base_df[['inn', 'join_type']].drop_duplicates()
join_stat = fallback_df.groupby('join_type', as_index=False).agg(inn_cnt=('inn', 'nunique')).sort_values('inn_cnt', ascending=False)

excel_eval = pd.DataFrame({'inn': sorted(excel_inn_set)}).merge(
    base_df[['inn', 'cdi_id', 'cft_id', 'contract_number_final', 'agr_id_final']].drop_duplicates(),
    on='inn',
    how='left'
)

excel_eval['has_cdi'] = excel_eval['cdi_id'].notna() & (excel_eval['cdi_id'].astype(str) != '')
excel_eval['has_cft'] = excel_eval['cft_id'].notna() & (excel_eval['cft_id'].astype(str) != '')
excel_eval['has_contract_final'] = excel_eval['contract_number_final'].notna() & (excel_eval['contract_number_final'].astype(str) != '')
excel_eval['has_agr_id_final'] = excel_eval['agr_id_final'].notna() & (excel_eval['agr_id_final'].astype(str) != '')

key_df = base_df[['inn', 'contract_number_final', 'agr_id_final']].copy()
key_df = key_df[key_df['inn'].notna() & key_df['contract_number_final'].notna()].copy()
key_dup_df = key_df.groupby(['inn', 'contract_number_final'], as_index=False).agg(
    rows=('agr_id_final', 'size'),
    agr_id_final_nunique=('agr_id_final', 'nunique')
)
key_dup_df = key_dup_df[(key_dup_df['rows'] > 1) | (key_dup_df['agr_id_final_nunique'] > 1)].sort_values(['rows', 'agr_id_final_nunique'], ascending=False)

conflict_df = key_df.groupby(['inn', 'contract_number_final'], as_index=False).agg(agr_id_final_nunique=('agr_id_final', 'nunique'))
conflict_df = conflict_df[conflict_df['agr_id_final_nunique'] > 1].sort_values('agr_id_final_nunique', ascending=False)

summary_df = pd.DataFrame([
    {'metric': 'excel_unique_inn', 'value': len(excel_inn_set)},
    {'metric': 'lake_unique_inn', 'value': len(lake_inn_set)},
    {'metric': 'inn_in_both', 'value': len(in_both_set)},
    {'metric': 'inn_only_in_excel', 'value': len(only_excel_set)},
    {'metric': 'inn_only_in_lake', 'value': len(only_lake_set)},
    {'metric': 'excel_inn_with_cdi', 'value': int(excel_eval['has_cdi'].sum())},
    {'metric': 'excel_inn_with_cft', 'value': int(excel_eval['has_cft'].sum())},
    {'metric': 'excel_inn_with_contract_final', 'value': int(excel_eval['has_contract_final'].sum())},
    {'metric': 'excel_inn_with_agr_id_final', 'value': int(excel_eval['has_agr_id_final'].sum())},
    {'metric': 'duplicate_key_cnt_inn_contract_final', 'value': len(key_dup_df)},
    {'metric': 'conflict_agr_id_final_cnt', 'value': len(conflict_df)},
])

summary_df

In [ ]:
examples_only_excel_df = pd.DataFrame({'inn': sorted(list(only_excel_set))[:5]})
examples_only_lake_df = pd.DataFrame({'inn': sorted(list(only_lake_set))[:5]})
examples_missing_contract_df = excel_eval[~excel_eval['has_contract_final']][['inn']].drop_duplicates().sort_values('inn').head(5).reset_index(drop=True)
examples_missing_agr_id_final_df = excel_eval[~excel_eval['has_agr_id_final']][['inn']].drop_duplicates().sort_values('inn').head(5).reset_index(drop=True)

examples_only_excel_df, examples_only_lake_df, examples_missing_contract_df, examples_missing_agr_id_final_df

In [ ]:
coverage_contract_pct = round(100.0 * int(excel_eval['has_contract_final'].sum()) / len(excel_eval), 2) if len(excel_eval) else 0.0
no_match_pct = round(100.0 * int((fallback_df['join_type'] == 'no_match').sum()) / len(fallback_df), 2) if len(fallback_df) else 0.0
dup_cnt = len(key_dup_df)

qc_gate_df = pd.DataFrame([
    {'check': 'contract_final_coverage_ge_95pct', 'value': coverage_contract_pct, 'threshold': 95.0, 'pass': coverage_contract_pct >= 95.0},
    {'check': 'no_match_le_5pct', 'value': no_match_pct, 'threshold': 5.0, 'pass': no_match_pct <= 5.0},
    {'check': 'duplicate_inn_contract_eq_0', 'value': dup_cnt, 'threshold': 0, 'pass': dup_cnt == 0},
])

qc_gate_df

In [ ]:
out_dir = Path(output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

summary_path = out_dir / 'preflight_summary_feb.xlsx'
join_stat_path = out_dir / 'preflight_join_type_stat_feb.xlsx'
qc_gate_path = out_dir / 'preflight_qc_gate_feb.xlsx'
duplicates_path = out_dir / 'preflight_duplicate_keys_feb.xlsx'
conflicts_path = out_dir / 'preflight_conflicts_agr_id_final_feb.xlsx'
examples_path = out_dir / 'preflight_examples_feb.xlsx'
base_path = out_dir / 'preflight_base_feb.parquet'

summary_df.to_excel(summary_path, index=False)
join_stat.to_excel(join_stat_path, index=False)
qc_gate_df.to_excel(qc_gate_path, index=False)
key_dup_df.to_excel(duplicates_path, index=False)
conflict_df.to_excel(conflicts_path, index=False)

examples_export_df = pd.concat([
    examples_only_excel_df.assign(example_type='only_excel'),
    examples_only_lake_df.assign(example_type='only_lake'),
    examples_missing_contract_df.assign(example_type='missing_contract_final'),
    examples_missing_agr_id_final_df.assign(example_type='missing_agr_id_final'),
], ignore_index=True)
examples_export_df.to_excel(examples_path, index=False)

base_df.to_parquet(base_path, index=False)

summary_path, join_stat_path, qc_gate_path, duplicates_path, conflicts_path, examples_path, base_path